# Predict Ain Km 4 -> ENET'Com with the ready XGBoost model

This notebook loads `ev_energy_model (1).json` from the same folder and predicts with the five features stored in that model.

In [ ]:
from pathlib import Path

import pandas as pd
import xgboost as xgb
from IPython.display import display

MODEL_PATH = Path.cwd() / 'ev_energy_model (1).json'
if not MODEL_PATH.is_file():
    raise FileNotFoundError(
        'Open this notebook from the EV energy consumption folder so the model JSON is beside it.'
    )

model = xgb.Booster()
model.load_model(MODEL_PATH)
FEATURES = model.feature_names
print('Model:', MODEL_PATH.name)
print('Required features:', FEATURES)

## Enter input values

The values below represent the Ain Km 4 -> ENET'Com trip. Edit them to test another scenario.

In [ ]:
# Trip context for reference; distance is NOT accepted by this model.
ORIGIN = 'Route El Ain Km 4, Sfax'
DESTINATION = "ENET'Com, Cite El Ons, Sakiet Ezzit, Sfax"
TRIP_DISTANCE_KM = 12.198

input_values = {
    'Speed_Smooth': 36.59,       # Distance-weighted average route speed, km/h
    'Road_Slope_pct': 0.0,       # Flat-road assumption
    'Ambient_Temp_C': 21.5,      # Destination weather example
    'Weight_kg': 1270.0,         # BMW i3 120 Ah reference mass
    'Acceleration_m_s2': 0.0,    # Steady-speed assumption
}

model_input = pd.DataFrame([input_values], columns=FEATURES)
assert list(model_input.columns) == FEATURES
assert model_input.notna().all().all()
prediction = float(model.predict(xgb.DMatrix(model_input))[0])

print(f'Trip: {ORIGIN} -> {DESTINATION}')
print('Reference distance:', TRIP_DISTANCE_KM, 'km (not used by this model)')
display(model_input)
print('Raw model prediction:', round(prediction, 6))

## Interpretation

For the supplied values, the model returns approximately **13.160264**. The JSON stores neither the training-target name nor its unit, so this notebook intentionally labels it a **raw prediction**, not kWh.

The model does not accept distance, battery state, voltage, humidity, wind, or tire pressure. Therefore it cannot directly predict total energy for a specific trip length unless its target was already defined as a rate such as energy per distance. Check the training code or metadata before using this value in the EV copilot.